# Dataset Preprocessing

Upon clustering the data and producing storms, we preprocessed our data according to the following steps:
+ *insert preprocessing steps here*

In this notebook, we continue to preprocess and clean our dataset.

In [1]:
import xarray as xr
import pandas as pd
import numpy as np

In [3]:
dataset = pd.read_hdf('datasets/landfalling_storm_quantities_df.h5')

First, notice that 87 of the storms have negative SLP gradient over the ocean at landfall. This means that that storm had to have never had a component over the ocean upon landfall. Let's exclude these.

In [6]:
(dataset.max_ocean_SLP_gradient < 0).sum()

np.int64(87)

In [7]:
dataset = dataset.loc[dataset.max_ocean_SLP_gradient >= 0]

Now, let's get the amount of precipitation in gigatons.

In [8]:
dataset['cumulative_rainfall_ais'] = dataset['cumulative_rainfall_ais']/(10**13)
dataset['cumulative_snowfall_ais'] = dataset['cumulative_snowfall_ais']/(10**13)

/tmp/ipykernel_452304/315284922.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['cumulative_rainfall_ais'] = dataset['cumulative_rainfall_ais']/(10**13)
/tmp/ipykernel_452304/315284922.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['cumulative_snowfall_ais'] = dataset['cumulative_snowfall_ais']/(10**13)


Let's also add the duration in hours.

In [9]:
dataset['duration'] = dataset['duration']/pd.Timedelta(1, 'h')

/tmp/ipykernel_452304/2145110790.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['duration'] = dataset['duration']/pd.Timedelta(1, 'h')


For fun, let's also rename the storms. Naming convention will be `YYYY-x` where `YYYY` is the year of formation and `x` is the xth storm of that year.

In [10]:
labels_lst = []
for year in dataset.start_date.dt.year.unique():

    n_storms = (dataset.start_date.dt.year == year).sum()
    year_labels = f'{year}_' + (np.arange(0, n_storms) + 1).astype(str)
    
    labels_lst = labels_lst + list(year_labels)

In [11]:
dataset.index = labels_lst
dataset.index.name = 'Label'

Save the cleaned dataset as another hdf5 file.

In [13]:
dataset.to_hdf('datasets/landfalling_df_cleaned.h5', key='df')

/tmp/ipykernel_452304/3342931258.py:1: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block3_values] [items->Index(['data_array', 'region', 'coarser_region', 'trajectory',
       'max_landfalling_v850hPa', 'avg_landfalling_v850hPa'],
      dtype='object')]

  dataset.to_hdf('datasets/landfalling_df_cleaned.h5', key='df')


Now, let's save this to a csv file that can be read into R later for the erfs.

In [21]:
saveable_cleaned = dataset.drop(columns=['data_array', 'is_landfalling', 'start_date', 'end_date', 'region', 'trajectory'])
saveable_cleaned.to_csv('datasets/landfalling_df_cleaned.csv')

In [24]:
# train/test split
np.random.seed(12345)
n = saveable_cleaned.shape[0]
inds = np.arange(0, n)
frac_test = 0.2
test_inds = np.random.choice(inds, size=int(n*frac_test), replace=False)
train_inds = np.setdiff1d(inds, test_inds, assume_unique=True)

train_data = saveable_cleaned.iloc[train_inds]
test_data = saveable_cleaned.iloc[test_inds]

train_data.to_csv('datasets/model_ready/train.csv')
test_data.to_csv('datasets/model_ready/test.csv')